# Case Study - Heart Disease

You have been asked by a hospital to use machine learning to predict heart disease. Your job is to develop a model and highlight two to three important features that doctors and nurses can focus on to improve patient health.

You decide to use a decision tree classifier with fine-tuned hyperparameters. After the model has been built, you will interpret results using **feature_importances_**, an attribute that determines the most important features in predicting heart disease.

We will use a modified version of Heart Disease dataset (https://archive.ics.uci.edu/ml/datasets/Heart+Disease) provided by the UCI Machine Learning Repository (https://archive.ics.uci.edu/ml/index.php) with null values cleaned up

In [11]:
# Import pandas and numpy
import pandas as pd
import numpy as np

# Import warnings
import warnings
warnings.filterwarnings('ignore')

In [28]:
pd.__version__

'2.2.2'

In [29]:
np.__version__

'2.0.2'

In [12]:
# Upload heart.csv to dataFrame
df_heart = pd.read_csv('heart_disease.csv')

# Show first five rows
df_heart.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


The target column, conveniently labeled 'target' is binary, with 1 indicating that the patient has heart disease and 0 indicating that they do not.

| Feature     | Description |
|-------------|-------------|
| `age`       | Age in years |
| `sex`       | Sex (`1` = male, `0` = female) |
| `cp`        | Chest pain type: <br>• `1` = Typical angina <br>• `2` = Atypical angina <br>• `3` = Non-anginal pain <br>• `4` = Asymptomatic |
| `trestbps`  | Resting blood pressure (in mm Hg) on admission to the hospital |
| `chol`      | Serum cholesterol in mg/dl |
| `fbs`       | Fasting blood sugar > 120 mg/dl (`1` = true, `0` = false) |
| `restecg`   | Resting ECG results: <br>• `0` = Normal <br>• `1` = ST-T wave abnormality <br>• `2` = Left ventricular hypertrophy (by Estes' criteria) |
| `thalach`   | Maximum heart rate achieved |
| `exang`     | Exercise induced angina (`1` = yes, `0` = no) |
| `oldpeak`   | ST depression induced by exercise relative to rest |
| `slope`     | Slope of peak exercise ST segment: <br>• `1` = Upsloping <br>• `2` = Flat <br>• `3` = Downsloping |
| `ca`        | Number of major vessels (0–3) colored by fluoroscopy |
| `thal`      | Thalassemia: <br>• `3` = Normal <br>• `6` = Fixed defect <br>• `7` = Reversible defect |


In [31]:
# Import scikit-learn to check its version
import sklearn
sklearn.__version__

'1.6.1'

---

***Split the data into training and test sets in preparation for machine learning***

---

In [13]:
# 🧪 Import train_test_split to partition data into training and testing sets
from sklearn.model_selection import train_test_split

In [14]:
# split data into X and y
X = df_heart.iloc[:,:-1]
y = df_heart.iloc[:,-1]

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=2)

In [15]:
# 🌳 Import the Decision Tree Classifier from scikit-learn
from sklearn.tree import DecisionTreeClassifier

# Import cross_val_score
from sklearn.model_selection import cross_val_score

---

***Before implementing hyperparameters, it's helpful to have a baseline model for comparison.***

---

In [16]:
# Initialize Decision Tree Classifier
model = DecisionTreeClassifier(random_state=2)

# Obtain scores of cross-validation
scores = cross_val_score(model, X, y, cv=5)

# Display accuracy
print('Accuracy:', np.round(scores, 2))

# Display mean accuracy
print('Accuracy mean: %0.2f' % (scores.mean()))

# Accuracy: [0.74 0.85 0.77 0.73 0.7 ]
# Accuracy mean: 0.76

Accuracy: [0.74 0.85 0.77 0.73 0.7 ]
Accuracy mean: 0.76


The initial accuracy is 76%. Let's see what gains can be made with hyperparameter fine-tuning.

---

***Hyperparameter Tuning using RandomizedSearchCV***

---

***Why RandomizedSearchCV vs GridSearchCV?***

When fine-tuning many hyperparameters, GridSearchCV can take too much time. The scikit-learn library provides RandomizedSearchCV as an alternative. RandomizedSearchCV works in the same way as GridSearchCV, but instead of trying all hyperparameters, it tries a random number of combinations. It's not meant to be exhaustive. It's meant to find the best combinations in limited time.


***Define a function that uses RandomizedSearchCV to return the best model along with the scores***

In [17]:
# Import RandomizedSearchCV to perform randomized hyperparameter search
from sklearn.model_selection import RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier  # Make sure this is also imported
from sklearn.metrics import accuracy_score       # Needed for evaluating test accuracy

# Define a function to perform randomized hyperparameter search
# - params: dictionary of hyperparameter options
# - runs: number of different combinations to try (default is 20)
# - clf: the base classifier model to tune (default is DecisionTreeClassifier)
def randomized_search_clf(params, runs=20, clf=DecisionTreeClassifier(random_state=2)):

    # Create a RandomizedSearchCV object
    # - clf: the model to tune
    # - params: hyperparameters to try
    # - n_iter: how many random combinations to test
    # - cv: 5-fold cross-validation (data is split into 5 parts)
    # - n_jobs: use all available cores for faster computation
    # - random_state: ensures reproducible results
    rand_clf = RandomizedSearchCV(clf, params, n_iter=runs,
                                  cv=5, n_jobs=-1, random_state=2)

    # Train the model using the training data
    rand_clf.fit(X_train, y_train)

    # Get the model with the best combination of hyperparameters
    best_model = rand_clf.best_estimator_

    # Get the best cross-validation accuracy score during training
    best_score = rand_clf.best_score_

    # Print the best training accuracy score
    print("Training score: {:.3f}".format(best_score))

    # Use the best model to predict labels on the test set
    y_pred = best_model.predict(X_test)

    # Calculate the accuracy of predictions on the test set
    accuracy = accuracy_score(y_test, y_pred)

    # Print the test accuracy
    print('Test score: {:.3f}'.format(accuracy))

    # Return the best model so it can be reused later
    return best_model


Choosing best hyperparameters - Experimentation is the name of the game - aim is to reduce variance

In [18]:
randomized_search_clf(params={

    'criterion': ['entropy', 'gini'],
    # Defines the function to measure the quality of a split.
    # 'gini' = Gini Impurity (default), 'entropy' = Information Gain.
    # Both guide how the tree chooses features to split on.

    'splitter': ['random', 'best'],
    # 'best' will always choose the best split among all features (greedy).
    # 'random' selects a random feature at each split point — useful for reducing overfitting.

    'min_weight_fraction_leaf': [0.0, 0.0025, 0.005, 0.0075, 0.01, 0.05],
    # The minimum fraction of the total weight (samples) required at a leaf node.
    # Especially useful when sample weights are used.
    # Prevents creating very small (unrepresentative) leaf nodes.

    'min_samples_split': [2, 3, 4, 5, 6, 8, 10],
    # The minimum number of samples required to split an internal node.
    # Higher values create broader, simpler trees (can prevent overfitting).

    'min_samples_leaf': [1, 0.01, 0.02, 0.03, 0.04],
    # Minimum number of samples required to be at a leaf node.
    # If a float, it represents a fraction of the training set.
    # Forces leaves to generalize better by having enough data.

    'min_impurity_decrease': [0.0, 0.0005, 0.005, 0.05, 0.10, 0.15, 0.2],
    # A node will be split only if the split decreases impurity by at least this threshold.
    # Useful for regularizing the tree and stopping splits that don’t help much.

    'max_leaf_nodes': [10, 15, 20, 25, 30, 35, 40, 45, 50, None],
    # Limits the number of leaf nodes (terminal nodes).
    # Fewer leaves = simpler trees; None = unlimited growth.

    'max_features': ['auto', 0.95, 0.90, 0.85, 0.80, 0.75, 0.70],
    # Number of features to consider when looking for the best split.
    # 'auto' = sqrt(n_features) for classification.
    # Fractions are useful for reducing variance and speeding up computation.

    'max_depth': [None, 2, 4, 6, 8],
    # Maximum depth of the tree.
    # None allows unlimited depth, but shallower trees tend to generalize better.

    # Note: 'min_weight_fraction_leaf' was included earlier and is repeated.
    # Python dictionaries can't have duplicate keys — this second mention will override the first.
    # So only the second list (with 0.05 included) will be used.
})


Training score: 0.798
Test score: 0.855


DecisionTreeClassifier(criterion='entropy', max_depth=8, max_features=0.8,
                       max_leaf_nodes=45, min_samples_leaf=0.04,
                       min_samples_split=10, min_weight_fraction_leaf=0.05,
                       random_state=2)

This is a definite improvement from baseline model, and the model generalizes well on the test set

***Improving hyperparameters by Narrowing the range***

### 🔧 Strategies to Improve Hyperparameter Tuning for Decision Trees

* **🔍 Narrow the range around promising values**

  * If `max_depth=8` gave the best result, try a tighter range like `max_depth = [7, 8, 9]` instead of `[2, 4, 6, 8, None]`.
  * This focuses search near the local optimum and saves compute time.

* **🚫 Eliminate unnecessary hyperparameters**

  * If the default value works well (e.g., `criterion='gini'`), you can **remove alternatives like `'entropy'`** to simplify the search.
  * Research and experience suggest little gain from `entropy` over `gini` in most datasets.

* **✅ Freeze hyperparameters that have little impact**

  * Parameters like `min_impurity_split` and `min_impurity_decrease` often don’t significantly improve performance and may be left at defaults (`0.0`).
  * Avoid tuning them unless your dataset is large and complex.

* **🎯 Prioritize high-impact parameters**

  * Focus more on tuning:

    * `max_depth`
    * `min_samples_split`
    * `min_samples_leaf`
    * `max_leaf_nodes`
  * These control **model complexity and overfitting**, and usually have the biggest influence on accuracy.

* **🧪 Use results from previous runs**

  * Inspect the top 5–10 models from `rand_clf.cv_results_` to see which values consistently perform well.
  * Use those to **narrow future hyperparameter ranges**.

* **⚖️ Balance exploration and compute time**

  * Early runs can be coarse (e.g., wide ranges, many values). Later runs should **zoom in on effective regions** of the hyperparameter space.
  * Limit `n_iter` when using many parameters; otherwise, increase `n_iter` if your space is small.

* **📊 Visualize performance trends**

  * Use plots to visualize how accuracy changes with `max_depth`, `min_samples_split`, etc.
  * This can highlight **non-linear trends** or flat regions in the parameter space.

* **📚 Refer to domain or model-specific defaults**

  * Scikit-learn’s defaults are reasonable for most small–medium datasets. If your dataset is highly imbalanced or large-scale, consider deviating.


In [19]:
# Second pass - new hyperparameter range with an increase of 100 runs
randomized_search_clf(
    params={
        'max_depth': [None, 6, 7],
        # max_depth limits how deep the tree can go.
        # Shallower trees generalize better; deeper trees may overfit.

        'max_features': ['auto', 0.78],
        # Number of features to consider at each split.
        # 'auto' (sqrt for classification) and 78% of total features.
        # Helps balance bias and variance.

        'max_leaf_nodes': [45, None],
        # Controls the maximum number of terminal nodes (leaves).
        # Fewer nodes = more general model.

        'min_samples_leaf': [1, 0.035, 0.04, 0.045, 0.05],
        # Minimum number (or fraction) of samples required in a leaf node.
        # Helps prevent overfitting by forcing leaves to have enough data.

        'min_samples_split': [2, 9, 10],
        # Minimum number of samples needed to split a node.
        # Higher values make the tree more conservative (prune more).

        'min_weight_fraction_leaf': [0.0, 0.05, 0.06, 0.07],
        # Fraction of total weight needed at a leaf.
        # Useful when using sample weights; helps avoid tiny leaf nodes.
    },
    runs=100  # Number of different random combinations to test
)

Training score: 0.802
Test score: 0.868


DecisionTreeClassifier(max_depth=7, max_features=0.78, max_leaf_nodes=45,
                       min_samples_leaf=0.045, min_samples_split=9,
                       min_weight_fraction_leaf=0.06, random_state=2)

✅ **NOTE**: This model is more accurate in the training and test score.

***Baseline Comparison using cross validation***

For a proper baseline of comparison, however, it's essential to put the new model into **cross_val_clf**

In [21]:
# 🌳 Import the Decision Tree Classifier from scikit-learn
from sklearn.tree import DecisionTreeClassifier

In [24]:
# Initialize Decision Tree Classifier

# The parameter values were obtained from previous run after hyperparameter tuning
model = DecisionTreeClassifier(
            class_weight=None,
            criterion='gini',
            max_depth=7,
            max_features=0.78,
            max_leaf_nodes=45,
            min_impurity_decrease=0.0,
            min_samples_leaf=0.045,
            min_samples_split=9,
            min_weight_fraction_leaf=0.06,
            random_state=2,
            splitter='best')

# Obtain scores of cross-validation
scores = cross_val_score(model, X, y, cv=5)

# Display accuracy
print('Accuracy:', np.round(scores, 2))

# Display mean accuracy
print('Accuracy mean: %0.2f' % (scores.mean()))

Accuracy: [0.82 0.9  0.8  0.8  0.78]
Accuracy mean: 0.82


This is six percentage points higher than the default model. When it comes to predicting heart disease, more accuracy can save lives.

***Saving the best model***

Define the model using the **best hyperparameters** and fit it on the **entire dataset**

In [25]:
best_clf = DecisionTreeClassifier(
              class_weight=None,
              criterion='gini',
              max_depth=7,
              max_features=0.78,
              max_leaf_nodes=45,
              min_impurity_decrease=0.0,
              min_samples_leaf=0.045,
              min_samples_split=9,
              min_weight_fraction_leaf=0.06,
              random_state=2,
              splitter='best')
best_clf.fit(X, y)

DecisionTreeClassifier(max_depth=7, max_features=0.78, max_leaf_nodes=45,
                       min_samples_leaf=0.045, min_samples_split=9,
                       min_weight_fraction_leaf=0.06, random_state=2)

In [26]:
best_clf.feature_importances_

array([0.04826754, 0.04081653, 0.48409586, 0.00568635, 0.        ,
       0.        , 0.        , 0.00859483, 0.        , 0.02690379,
       0.        , 0.18069065, 0.20494446])

### 🎯 What does `best_clf.feature_importances_` show?

It gives an array of **feature importance scores**, one for each input feature (column) in the same order as `X.columns`.

These scores tell you **how important each feature is** in making predictions for your trained **decision tree**.

---

### ✅ Key Points to Understand:

* **Higher value = More important**
  A higher number means the feature contributed more to how the decision tree made splits (i.e., helped reduce impurity).

* **Zero value = Not used**
  A value of `0.0` means the feature **was not used at all** in any split.

* **All values add up to 1**
  The importance scores are **normalized**, so the total of all values will be **1.0**.

---

### 🧠 Example Interpretation

Let’s pair these importance values with feature names (assuming `X.columns` in this order):

```python
X.columns:
['age', 'sex', 'cp', 'trestbps', 'chol',
 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak',
 'slope', 'ca', 'thal']
```

Now let’s interpret the most important features:

```plaintext
Feature           Importance    Meaning
-----------------------------------------
cp                0.4841        🟢 Chest pain type — the most decisive feature!
thal              0.2049        🟡 Thalassemia test result — second most important
ca                0.1807        🟡 Number of vessels colored — third most important
age               0.0483        🟠 Minor influence
sex               0.0408        🟠 Minor influence
oldpeak           0.0269        🟠 Slight role
thalach           0.0086        🔵 Barely used
Others            0.0           ⚪️ Not used in any decision splits
```

---

### 🧪 What can you do with this?

* **Feature selection**: You might drop or deprioritize features with **zero importance** to simplify your model.
* **Explainability**: You can explain predictions better by pointing to which features matter most.
* **Model debugging**: If an expected feature has low importance, investigate data quality or feature encoding.

In [27]:
# Zip column names (features) and their corresponding importance scores into a dictionary
# - X.columns gives the list of column (feature) names
# - best_clf.feature_importances_ gives the importance score for each feature (same order)
# - zip() combines them into pairs like: ('age', 0.12), ('sex', 0.02), ...
# - dict() converts these pairs into a dictionary
feature_dict = dict(zip(X.columns, best_clf.feature_importances_))

# Import the operator module so we can use itemgetter to sort by values in a dictionary
import operator

# Sort the dictionary by importance values (descending order)
# - feature_dict.items() turns the dictionary into a list of (key, value) tuples
# - key=operator.itemgetter(1) tells Python to sort by the second item in each tuple (i.e., the importance score)
# - reverse=True sorts in descending order (highest importance first)
# - [0:3] slices the list to get only the top 3 most important features
sorted(feature_dict.items(), key=operator.itemgetter(1), reverse=True)[0:3]


[('cp', np.float64(0.4840958610240171)),
 ('thal', np.float64(0.20494445570568706)),
 ('ca', np.float64(0.18069065321397942))]

The three most important features are as follows:

*   **'cp'**: Chest pain type (1 = typical angina, 2 = atypical angina, 3 = non-anginal pain, 4 = asymptomatic)
*   **'thalach'**: Maximum heart rate achieved
*   **'ca'**: Number of major vessels (0-3) colored by fluoroscopy

These numbers indicate how much each feature contributed to the tree's ability to split the data and reduce uncertainty (impurity). For example, cp (chest pain type) contributed about 48% of the total decision-making power of the model.

You can tell the doctors and nurses that your model predicts if the patient has a heart disease with 82% accuracy using chest pain, maximum heart rate, and fluoroscopy as the three most important characteristics.

### 🏥 Explaining Your Model to Medical Staff

**What makes the biggest difference in predicting heart disease?**

Your model has learned patterns from patient data and identified the **top 3 most important characteristics**:

1. **Chest pain type** (`cp`)
   (e.g., typical angina, non-anginal pain, asymptomatic)
   👉 This feature alone guided **almost half** of the model’s decision-making.

2. **Thalassemia test result** (`thal`)
   (whether the heart defect is normal, fixed, or reversible)

3. **Number of major vessels** (`ca`)
   (visible through fluoroscopy — 0 to 3 vessels)

---

### ✅ How to interpret this

You can think of these features as the **most informative clues** the model uses to predict if a patient has heart disease.

* **Chest pain type** was the strongest signal — **it mattered more than all other features combined**.
* The model also strongly relies on **thalassemia results** and **fluoroscopy vessel count**.
* Other things like cholesterol or heart rate were less helpful in this specific model.

---

### 🧠 Practical Summary

> This model predicts heart disease with about **82% accuracy**.
> The most important inputs are:
> **chest pain symptoms**, **thalassemia test result**, and **vessel imaging**.

This helps clinicians focus on what matters most — and also improves trust in the model.